# 🎓 Deep Reinforcement Learning (DQN & Double DQN): Complete Study Guide & Solutions

**Course:** Data & A.I. 6 (DAI6) — Week 2 / Lecture 3  
**Instructor:** Alexander Michielsen  
**Based on:** PyTorch DQN Tutorial by [Adam Paszke](https://github.com/apaszke) & [Mark Towers](https://github.com/pseudo-rnd-thoughts)  
**Format:** Simple Words, Visual Tables, Step-by-Step Math & Fully Solved `#TODO Student` Code

---

## 🧭 Quick Refresh: Why Move from Week 1 (Q-Table) to Week 2 (Deep Q-Network)?

In **Week 1**, we used a **Q-Table** (a giant spreadsheet) where every unique game state had its own row. That worked great on a tiny $4 \times 3$ Gridworld, but failed completely on `mediumGrid` Pacman because there were over $10^{15}$ states.

In **`CartPole-v1`**, the state is made of **4 continuous decimal numbers** (Cart Position, Cart Velocity, Pole Angle, Pole Angular Velocity). Because there are infinitely many decimal numbers, **you cannot build a table with infinite rows!**

| Feature | Week 1: Tabular Q-Learning | Week 2: Deep Q-Network (DQN) |
| :--- | :--- | :--- |
| **How Q-Values Are Stored** | A lookup table (`self.Q = util.Counter()`). | A **Neural Network** (`DQN(nn.Module)`) that predicts Q-values from state numbers. |
| **Can Handle Continuous States?** | ❌ **No** (infinite rows needed). | ✅ **Yes** (takes any 4 decimal numbers as input). |
| **Generalization** | ❌ **None** (State A tells you nothing about nearby State B). | ✅ **High** (similar states automatically get similar Q-values). |
| **Core Update Idea** | $Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$ | Train neural network weights $\theta$ via gradient descent so $Q_\theta(s,a)$ matches $r + \gamma \max_{a'} Q_{\theta^-}(s',a')$. |

In [ ]:
# For tips on running notebooks in Google Colab, see
# https://pytorch.org/tutorials/beginner/colab
%matplotlib inline

---

## 🎮 1. The Task: `CartPole-v1` Explained Simply

Imagine balancing a broomstick upright on your hand. To keep it from falling, you move your hand **left** or **right**.

![CartPole](https://pytorch.org/tutorials/_static/img/cartpole.gif)

| Environment Element | Value in `CartPole-v1` | What It Means in Simple Words |
| :--- | :--- | :--- |
| **Observation / State (`n_observations = 4`)** | `[cart_pos, cart_vel, pole_angle, pole_vel]` | 4 decimal numbers describing where the cart is and how fast the pole is tipping. |
| **Action Space (`n_actions = 2`)** | `0` = Push Left, `1` = Push Right | The only 2 buttons the agent can press. |
| **Reward (`+1` per step)** | `+1.0` for every timestep the pole stays upright | The longer you keep the pole balanced, the higher your score! |
| **Episode End (`terminated` vs `truncated`)** | • **Terminated (Crash):** Pole tilts $> 12^\circ$ or cart moves $> 2.4$ units from center.<br/>• **Truncated (Win!):** Reaches **500 steps**. | The **maximum possible score** in one episode of `CartPole-v1` is **500**. |

### Packages Needed
If `gymnasium` is not installed yet, run `pip install gymnasium[classic_control]`.

In [ ]:
import gymnasium as gym
import math
import random
import matplotlib
import matplotlib.pyplot as plt
from collections import namedtuple, deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

env = gym.make("CartPole-v1")

# set up matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

# Automatically use NVIDIA GPU (cuda), Apple Silicon GPU (mps), or CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Using device: {device}")

---

## 🧠 2. Replay Memory (✅ Solved `#TODO Student #1`)

### Why Do We Need Replay Memory?
If a neural network only trains on consecutive frames as the cart moves right, all the training examples look almost identical! The network quickly **overfits** to moving right and **forgets** how to move left (**Catastrophic Forgetting**).

#### 💡 The Flashcard Analogy:
* **Without Replay Memory:** Studying only Page 10 of your textbook 100 times in a row, then forgetting Page 1.
* **With Replay Memory (`ReplayMemory`):** Writing every experience $(s, a, s', r)$ onto a flashcard, tossing it into a box of 10,000 flashcards (`deque(maxlen=10000)`), and **drawing 128 random flashcards** (`random.sample`) at each step!

| Class / Method | What It Does |
| :--- | :--- |
| `Transition` | A named tuple bundling `('state', 'action', 'next_state', 'reward')` together like a single flashcard. |
| `push(*args)` | Adds a new transition to the memory buffer. If the buffer is full (`> 10,000`), the oldest memory is automatically dropped. |
| **`sample(batch_size)`** | **✅ TODO #1 Solution:** Uses `random.sample(self.memory, batch_size)` to pick `batch_size` random transitions without replacement. |

In [ ]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))


# ✅ SOLVED TODO #1: Implement the sample method in the ReplayMemory class
class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """Randomly sample a batch of transitions from memory"""
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

---

## 📐 3. The DQN Algorithm & Neural Network (✅ Solved `#TODO Student #2`)

### How DQN Learns in 3 Simple Equations:

| Concept | Formula | Meaning in Simple Words |
| :--- | :--- | :--- |
| **1. Optimal Policy ($\pi^*$)** | $\pi^*(s) = \arg\max_a Q^*(s, a)$ | Pick the action (`Left` or `Right`) with the highest predicted Q-value. |
| **2. Temporal Difference (TD) Error ($\delta$)** | $\delta = Q(s, a) - \big(r + \gamma \max_{a'} Q(s', a')\big)$ | The gap between **what our network predicted** ($Q(s,a)$) and **the updated target** ($r + \gamma \max Q(s',a')$). |
| **3. Huber Loss ($\mathcal{L}(\delta)$)** | $\begin{cases} \frac{1}{2}\delta^2 & \text{if } |\delta| \le 1 \\ |\delta| - \frac{1}{2} & \text{otherwise} \end{cases}$ | A "shock-absorbing" loss function: acts like Squared Error when $\delta$ is small, and Absolute Error when $\delta$ is huge so noisy targets don't blow up gradients! |

---

### Building the `DQN` Neural Network (`✅ TODO #2`)

The exercise asks for **3 Linear layers with ReLU activations**:
1. **`self.layer1`**: Input = `n_observations` (`4`), Output = `128`, followed by `F.relu`.
2. **`self.layer2`**: Input = `128`, Output = `128`, followed by `F.relu`.
3. **`self.layer3`**: Input = `128`, Output = `n_actions` (`2` — one Q-value for Left, one for Right).

> ⚠️ **Two Important Notes for Students:**
> 1. In PyTorch's `torch.nn.functional as F`, the function is lowercase **`F.relu(...)`** (not `F.ReLU`).
> 2. **Do NOT put `F.relu` on `layer3`!** The final layer outputs raw Q-values, which must be allowed to take any real value (positive or negative).

In [ ]:
class DQN(nn.Module):
    # ✅ SOLVED TODO #2: 3 linear layers (n_observations -> 128 -> 128 -> n_actions) with ReLU activation
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

---

## ⚙️ 4. Hyperparameters, Two Networks (`policy_net` & `target_net`), and Action Selection

### Why Do We Create TWO Networks (`policy_net` and `target_net`)?
If we used a single network to both predict $Q(s,a)$ and calculate the target $r + \gamma \max Q(s', a')$, every time we updated the weights, the target would move too! That is like **a dog chasing its own tail**.
* **`policy_net` (The Student):** Trains at every single step and chooses actions.
* **`target_net` (The Answer Key):** Starts as an exact clone of `policy_net`, and is updated very slowly using **Soft Updates** (`TAU = 0.005`, meaning it moves only $0.5\%$ toward `policy_net` each step).

### Summary of Hyperparameters:

| Hyperparameter | Value | Simple Meaning |
| :--- | :---: | :--- |
| **`BATCH_SIZE`** | `128` | Number of random flashcards sampled from `ReplayMemory` per training step. |
| **`GAMMA` ($\gamma$)** | `0.99` | Discount factor (high value = agent cares strongly about future survival). |
| **`EPS_START` $\to$ `EPS_END`** | `0.9` $\to$ `0.05` | Exploration rate $\varepsilon$ starts at $90\%$ random moves and decays to $5\%$. |
| **`EPS_DECAY`** | `1000` | Controls how fast $\varepsilon$ shrinks (after 1,000 steps, $\varepsilon \approx 0.36$). |
| **`TAU` ($\tau$)** | `0.005` | Soft update speed for `target_net`: $\theta' \leftarrow 0.005\,\theta + 0.995\,\theta'$. |
| **`LR`** | `1e-4` | Learning rate ($0.0001$) for the `AdamW` optimizer. |

In [ ]:
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TAU = 0.005
LR = 1e-4

# Get number of actions from gym action space (2 for CartPole)
n_actions = env.action_space.n
# Get the number of state observations (4 for CartPole)
state, info = env.reset()
n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayMemory(10000)

steps_done = 0


def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
                    math.exp(-1. * steps_done / EPS_DECAY)  # Exponential decay of epsilon
    steps_done += 1
    if sample > eps_threshold:  # Exploit: pick action with largest Q-value
        with torch.no_grad():
            # t.max(1) returns (values, indices) of largest column per row.
            return policy_net(state).max(1).indices.view(1, 1)
    else:  # Explore: pick a random action (0 or 1)
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)


episode_durations = []


def plot_durations(show_result=False):
    plt.figure(1)
    durations_t = torch.tensor(episode_durations, dtype=torch.float)
    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')
    plt.xlabel('Episode')
    plt.ylabel('Duration')
    plt.plot(durations_t.numpy())
    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    plt.pause(0.001)  # pause a bit so that plots are updated
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

---

## 🔄 5. Optimization Step: Vanilla DQN vs. Double DQN (✅ Solved `#TODO #3` & `#TODO #5`)

### Step-by-Step Breakdown of `optimize_model()`:

| Step | Code Line | What It Does in Plain English |
| :---: | :--- | :--- |
| **1** | `transitions = memory.sample(BATCH_SIZE)` | **✅ TODO #3:** Grabs `128` random flashcards from `ReplayMemory`. |
| **2** | `batch = Transition(*zip(*transitions))` | Regroups the 128 flashcards into 4 lists: all 128 `states`, all 128 `actions`, all 128 `next_states`, and all 128 `rewards`. |
| **3** | `non_final_mask = ...` | Checks which transitions did **not** end in a crash (`next_state is not None`). Crashed states automatically get a future value of `0.0`. |
| **4** | `state_action_values = policy_net(state_batch).gather(1, action_batch)` | Computes $Q(s_t, a_t)$: runs all 128 states through `policy_net` and selects the Q-value of the specific action that was actually taken. |
| **5** | `next_state_values[non_final_mask] = ...` | Computes $V(s_{t+1})$ using either **Vanilla DQN** or **Double DQN** (see below!). |
| **6** | `expected_state_action_values = (next_state_values * GAMMA) + reward_batch` | Computes the Bellman target: $r_t + \gamma V(s_{t+1})$. |
| **7** | `loss = criterion(...)` & `optimizer.step()` | Computes Huber loss (`SmoothL1Loss`), clips gradients to `100` for stability, and updates `policy_net` weights! |

---

### 🔍 Understanding `#TODO #5`: Vanilla DQN vs. Double DQN

#### Why Does Vanilla DQN Struggle? (Maximization Bias)
In **Vanilla DQN**, the target network both **chooses** the best next action and **evaluates** its value using `.max(1).values`:
$$V_{\text{DQN}}(s_{t+1}) = \max_{a'} Q(s_{t+1}, a'; \theta^-)$$
Because neural network predictions always have some random noise, taking the `max` over noisy numbers systematically picks **overestimated** values!

#### How Does Double DQN Fix This?
**Double DQN** splits the job between our two networks:
1. **`policy_net` ($\theta$) SELECTS the best action index:**
   $$a^* = \arg\max_{a'} Q(s_{t+1}, a'; \theta) \quad \longrightarrow \quad \text{next\_state\_actions = policy\_net(non\_final\_next\_states).max(1).indices.unsqueeze(1)}$$
2. **`target_net` ($\theta^-$) EVALUATES the Q-value of that chosen action:**
   $$V_{\text{DoubleDQN}}(s_{t+1}) = Q(s_{t+1}, a^*; \theta^-) \quad \longrightarrow \quad \text{target\_net(non\_final\_next\_states).gather(1, next\_state\_actions).squeeze(1)}$$

*(Toggle `USE_DOUBLE_DQN = True` or `False` below to compare both methods!)*

In [ ]:
# Toggle between Vanilla DQN (False) and Double DQN (True)
USE_DOUBLE_DQN = True

def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    
    # ✅ SOLVED TODO #3: Sample a batch of transitions from the memory
    transitions = memory.sample(BATCH_SIZE)
    
    # Transpose the batch: converts batch-array of Transitions to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))

    # Compute a mask of non-final states and concatenate the batch elements
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                            batch.next_state)), device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                       if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) using policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Compute V(s_{t+1}) for all next states.
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    
    # ✅ SOLVED TODO #5: Implement Double Q-Learning
    with torch.no_grad():
        if USE_DOUBLE_DQN:
            # Step 1: policy_net selects the best action indices (shape: [num_non_final, 1])
            next_state_actions = policy_net(non_final_next_states).max(1).indices.unsqueeze(1)
            # Step 2: target_net evaluates the Q-value for those selected actions
            next_state_values[non_final_mask] = target_net(non_final_next_states).gather(1, next_state_actions).squeeze(1)
        else:
            # Vanilla DQN: target_net both selects and evaluates via .max(1).values
            next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    # Compute the expected Q values (Bellman Target: r + gamma * V(s_{t+1}))
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

---

## 🏋️ 6. Main Training Loop (✅ Solved `#TODO Student #4`)

In each episode:
1. Reset the environment and get initial `state`.
2. Pick an `action` using $\varepsilon$-greedy (`select_action(state)`).
3. Step the environment to get `observation`, `reward`, `terminated`, `truncated`.
4. **✅ TODO #4:** Save `(state, action, next_state, reward)` into `memory` using `memory.push(state, action, next_state, reward)`.
5. Run one training batch (`optimize_model()`).
6. Soft-update `target_net` weights toward `policy_net` weights ($\theta' \leftarrow \tau \theta + (1 - \tau)\theta'$).

In [ ]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = 600
else:
    num_episodes = 50

for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state, info = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    for t in count():
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward], device=device)
        done = terminated or truncated

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # ✅ SOLVED TODO #4: Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        optimize_model()

        # Soft update of the target network's weights:
        # θ′ ← τ θ + (1 − τ)θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        target_net.load_state_dict(target_net_state_dict)

        if done:
            episode_durations.append(t + 1)
            plot_durations()
            break

print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()

---

## 📊 7. Overall Data Flow & Answers to Bonus Exercise Questions (3.1 & 3.2)

![](https://pytorch.org/tutorials/_static/img/reinforcement_learning_diagram.jpg)

### Answers to the Questions in `RL_Exercises_Week2.pdf` (Section 3: Bonus Exercises):

| Question from Exercise Sheet | Clear, Simple Answer |
| :--- | :--- |
| **1. Does Vanilla DQN always reach a duration of 500 within 600 episodes? Is the graph always the same?** | **No!** Reinforcement Learning is highly stochastic (random initial neural network weights, random starting pole angles, random $\varepsilon$-greedy actions, and random replay sampling). Different runs produce very different graphs, and Vanilla DQN sometimes fails to reach 500 within 600 episodes or suddenly drops after reaching 500. |
| **2. Why is Vanilla DQN training unstable even with `ReplayMemory` and Soft Updates?** | Because of **Maximization Bias**: in `target_net(non_final_next_states).max(1).values`, the same network both picks the best action and scores it. Whenever the network has random positive noise on an action, `.max(1)` selects that overestimated value, causing Q-values to inflate and destabilize the policy. |
| **3. Why does Double DQN make the training process more stable?** | **Double DQN** decouples **action selection** (`policy_net` picks the action index $a^* = \arg\max_{a'} Q_\theta(s', a')$) from **action evaluation** (`target_net` computes $Q_{\theta^-}(s', a^*)$). Because the two networks don't share the exact same instantaneous noise spikes, Q-value overestimation is drastically reduced, leading to a smoother, more reliable climb to 500 steps! |